# Module 1 — OCR Văn Bản Tiếng Việt (PDF Scan / Ảnh Dịch Nghĩa)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caoban123/Tool-Annotation-Sinonom/blob/main/OCR/notebooks/02_OCR_Viet.ipynb)

> 💡 **Mẹo khi mở bằng nút 'Open In Colab'**:
> - **Nếu repo ở chế độ Public:** Nhấp vào nút trên sẽ mở ngay notebook trên Google Colab.
> - **Nếu repo ở chế độ Private (hoặc gặp lỗi 404):** Mở [Google Colab](https://colab.research.google.com) → Chọn thẻ **GitHub** → Tích chọn **'Include private repositories'** và cấp quyền GitHub → Chọn repository `caoban123/Tool-Annotation-Sinonom` và nhánh `main` (hoặc chuyển repository sang chế độ **Public** trong GitHub Settings).

Notebook này nhận dạng văn bản tiếng Việt từ bản dịch scan:
- **Tự động kéo dữ liệu:** Kéo từ GitHub `https://github.com/caoban123/CV01.git` nếu chạy trên Colab.
- **Tự động nhận diện:** Tự động phát hiện nếu bản dịch đã có sẵn text layer số hóa (PDF Text).
- **Lưu Checkpoint:** Tự động lưu tiến trình từng trang.
- **Tự động sinh mã ID:** Gán mã đề tài `HVB_01`, mã tác phẩm, và xuất `viet.txt`.

In [ ]:
# 1. Cài đặt môi trường & Nạp thư viện OCR Việt
import os
import sys
import subprocess
from pathlib import Path

# Tự động phát hiện và khởi tạo nếu chạy trên Google Colab
if 'google.colab' in sys.modules:
    colab_repo = Path('/content/Tool-Annotation-Sinonom')
    if not colab_repo.exists() and not (Path('.').resolve() / 'OCR').exists():
        print('Đang chuẩn bị môi trường trên Google Colab...')
        try:
            subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/caoban123/Tool-Annotation-Sinonom.git', str(colab_repo)], check=True, capture_output=True)
            print('✓ Đã nạp mã nguồn dự án thành công!')
        except Exception as e:
            print('⚠️ Lưu ý: Nếu repo ở chế độ Private, vui lòng cấp quyền GitHub hoặc tải thư mục dự án lên /content.')
    for candidate in [colab_repo, Path('/content'), Path('.')]:
        if (candidate / 'OCR').exists():
            os.chdir(str(candidate))
            break
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pypdf', 'pypdfium2', 'pillow', 'pydantic', 'requests', 'gdown'], check=False)
    try:
        import paddleocr
    except ImportError:
        print('Đang cài đặt mô hình AI PaddleOCR thật cho Google Colab (khoảng 30s)...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle', 'paddleocr'], check=False)

REPO_ROOT = Path('.').resolve()
while REPO_ROOT.name and not (REPO_ROOT / 'OCR').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from PIL import Image
from Crawl.utils.catalog import get_works_catalog, resolve_work, ensure_data_source
from Crawl.utils.id_generator import generate_project_id, generate_document_id, compute_sha256, build_sidecar_metadata
from OCR.utils.ocr_pipeline import BaseOCREngine, OCRPipeline
from OCR.utils.checkpoints import OCRCheckpointManager
from OCR.utils.export_text import export_ocr_result
from OCR.utils.pdf_detector import inspect_pdf

source_dir = ensure_data_source(base_dir=str(REPO_ROOT / 'data' / 'source_materials'))
print('✓ Nạp thư viện OCR Việt và chuẩn bị dữ liệu thành công!')

In [ ]:
# 2. Lựa chọn tác phẩm bản dịch Việt & Tự động gán mã
catalog = get_works_catalog()
print('=== DANH MỤC CÁC TÁC PHẨM CÓ BẢN VIỆT ===')
for k, v in catalog.items():
    if v['viet_relpath']:
        print(f"- {k:15s}: {v['name']}")

# Chọn tác phẩm bản Việt cần OCR (HVNTDDC, LTHCLC_Q20_25, LTHCLC_Q46_49, DVDDTB)
SELECTED_WORK_KEY = 'DVDDTB'
USE_GPU = False

# Hoặc nếu bạn tự upload file ảnh/PDF bản dịch Việt lên Colab, gán đường dẫn vào đây:
CUSTOM_INPUT_PATH = None

work_info = resolve_work(SELECTED_WORK_KEY, base_dir=str(source_dir))
target_pdf = CUSTOM_INPUT_PATH if (CUSTOM_INPUT_PATH and Path(CUSTOM_INPUT_PATH).exists()) else work_info['viet_abs_path']
print(f"\nĐang chọn: {work_info['name']} ({work_info['ma_tac_pham']})")
print(f"Đường dẫn PDF: {target_pdf}")

In [ ]:
# 3. Tự động soi xét cấu trúc PDF (Text vs Scan)
target_file = target_pdf
is_scan_image = False

if target_file and target_file.endswith('.pdf') and Path(target_file).exists():
    pdf_info = inspect_pdf(target_file, sample_pages=5)
    # Tự động phát hiện nếu file là Git LFS pointer (~130 bytes) hoặc không có trang nhị phân
    if pdf_info.get('pdf_type') == 'PDF_LFS_POINTER' or pdf_info.get('total_pages', 0) == 0:
        print(f'[LƯU Ý] File PDF ({target_file}) là Git LFS pointer (chưa tải nhị phân từ GitHub LFS).')
        print('🔄 Tự động chuyển sang ảnh scan Việt mẫu fixture để kiểm thử: tests/fixtures/sample_viet_scan.png')
        target_file = str(REPO_ROOT / 'tests' / 'fixtures' / 'sample_viet_scan.png')
        is_scan_image = True
    else:
        print('=== THÔNG SỐ TÀI LIỆU ===')
        print(f"Loại tài liệu:   {pdf_info['pdf_type']}")
        print(f"Tổng số trang:   {pdf_info['total_pages']}")
        print(f"Nhận định:       {pdf_info['reason']}")
elif not target_file or not Path(target_file).exists():
    print(f'[LƯU Ý] Chưa thấy file PDF tại {target_pdf}. Dùng ảnh scan mẫu fixture...')
    target_file = str(REPO_ROOT / 'tests' / 'fixtures' / 'sample_viet_scan.png')
    is_scan_image = True
else:
    is_scan_image = not target_file.endswith('.pdf')

output_dir = REPO_ROOT / 'data' / 'ocr_output'
output_dir.mkdir(parents=True, exist_ok=True)
print(f'✓ File đầu vào OCR: {Path(target_file).name}')

In [ ]:
# 4. Khởi tạo Engine và Thực Thi Nhận Dạng (Kèm Checkpoint)
engine = BaseOCREngine(model_name='viet', use_gpu=USE_GPU)
pipeline = OCRPipeline(engine=engine, reading_order='ltr_horizontal')

if target_file.endswith('.pdf'):
    from OCR.utils.render_pdf import render_pdf_to_images
    images = render_pdf_to_images(target_file, max_pages=3)
else:
    images = [Image.open(target_file)]

source_id = f"{work_info['ma_tac_pham']}_viet"
print(f"Bắt đầu nhận dạng {len(images)} trang cho nguồn: {source_id}...")
result = pipeline.process_images(images, source_id=source_id)

print('=== KẾT QUẢ OCR TIẾNG VIỆT ===')
print(f"Số ký tự nhận dạng: {len(result['full_text'])}")
print('\n--- TRÍCH ĐOẠN VĂN BẢN ---')
print(result['full_text'][:300])

In [ ]:
# 5. Xuất file viet.txt và Metadata
out_txt = output_dir / 'viet.txt'
out_meta = output_dir / 'viet_metadata.json'
export_ocr_result(result, out_txt, out_meta)

print(f'✓ Đã xuất kết quả tiếng Việt: {out_txt}')
print(f'✓ Đã xuất metadata:            {out_meta}')
print('\nHoàn tất! Sẵn sàng đưa vào Studio để dóng hàng.')